In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df = pd.read_csv("data/campuspulse_akgec_cleaned.csv")

print(df.shape)
df.head()

(200000, 37)


,complaint_id,complaint_date,complaint_time,complainant_type,department,year_of_study,campus_zone,building,floor,room_lab_no,...,estimated_repair_cost,weather_condition,submission_channel,complaint_status,response_time_hours,resolution_time_hours,maintenance_team,resolution_type,priority,actual_expenditure
0,CP-0000001,2025-10-09,12:49:29,Student,Mechanical Engineering,4.0,Main Gate Zone,EN / EC Block,2,Faculty Room,...,2814,Cloudy,Admin Dashboard,Closed,9.99,12.68,Cleaning & Sanitation,Preventive Maintenance,Medium,7190
1,CP-0000002,2024-01-08,12:21:32,Student,Computer Science and Engineering (Artificial I...,4.0,Administrative Block,CSIT Block,Ground,AIML Lab-2,...,10427,Cloudy,Web Portal,Closed,16.30,14.66,IT Support,Software Fix,Low,23112
2,CP-0000003,2026-07-06,16:44:02,Maintenance Staff,Mechanical Engineering,NaN,Laboratory Zone,NLT Block,1,Classroom-301,...,34833,Rainy,Help Desk,Closed,3.07,23.98,Civil Maintenance,Repaired,Medium,71277
3,CP-0000004,2023-12-17,16:03:21,Student,Electronics and Communication Engineering,1.0,Administrative Block,LT Block,1,Classroom-11,...,2436,Cool,Help Desk,Closed,22.66,14.10,General Maintenance,Replaced,Low,6471
4,CP-0000005,2024-01-23,16:38:57,Administrative Staff,Computer Science and Engineering,NaN,Administrative Block,NLT Block,Ground,Classroom-302,...,16224,Humid,Help Desk,Resolved,5.97,25.78,Network Team,Repaired,High,42486


In [3]:
df["complaint_date"] = pd.to_datetime(df["complaint_date"],errors="coerce")

In [4]:
df["complaint_date"].dtype

dtype('<M8[us]')

In [6]:
df["complaint_month"] = df["complaint_date"].dt.month
df["complaint_dayofweek"] = df["complaint_date"].dt.dayofweek
df["complaint_year"] = df["complaint_date"].dt.year

In [7]:
df["complaint_time"].head()

0    12:49:29
1    12:21:32
2    16:44:02
3    16:03:21
4    16:38:57
Name: complaint_time, dtype: str

In [8]:
df["complaint_time"] = pd.to_datetime(df["complaint_time"].astype(str),errors="coerce")

C:\Users\acer\AppData\Local\Temp\ipykernel_21220\1033026776.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["complaint_time"] = pd.to_datetime(df["complaint_time"].astype(str),errors="coerce")


In [9]:
df["complaint_hour"] = df["complaint_time"].dt.hour

In [10]:
df.drop(columns=["complaint_date", "complaint_time"],inplace=True)

In [11]:
leakage_cols = ["complaint_status", "response_time_hours", "resolution_time_hours", "maintenance_team", "resolution_type", "actual_expenditure"]

In [12]:
df.drop(columns=leakage_cols, inplace=True)

In [13]:
df.drop(columns=["complaint_id"], inplace=True)

In [14]:
print(df.shape)
print(df.columns.tolist())

(200000, 32)
['complainant_type', 'department', 'year_of_study', 'campus_zone', 'building', 'floor', 'room_lab_no', 'facility_type', 'class_section', 'issue_category', 'issue_subcategory', 'severity', 'affected_users', 'issue_duration_hours', 'previous_similar_complaints', 'complaints_last_7_days', 'complaints_last_30_days', 'recurrence_count', 'equipment_age_years', 'infrastructure_age_years', 'connectivity_impact', 'safety_risk', 'academic_impact', 'operational_impact', 'estimated_repair_cost', 'weather_condition', 'submission_channel', 'priority', 'complaint_month', 'complaint_dayofweek', 'complaint_year', 'complaint_hour']


In [15]:
X = df.drop(columns="priority")
y = df["priority"]

In [16]:
print("X:", X.shape)
print("y:", y.shape)

X: (200000, 31)
y: (200000,)


In [17]:
print(y.value_counts())

priority
Medium      70197
Low         68808
High        42242
Critical    18753
Name: count, dtype: int64


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=42,stratify=y)

In [19]:
print(X_train.shape)
print(X_test.shape)

(160000, 31)
(40000, 31)


In [20]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()

print(num_cols)

['year_of_study', 'affected_users', 'issue_duration_hours', 'previous_similar_complaints', 'complaints_last_7_days', 'complaints_last_30_days', 'recurrence_count', 'equipment_age_years', 'infrastructure_age_years', 'estimated_repair_cost', 'complaint_month', 'complaint_dayofweek', 'complaint_year', 'complaint_hour']


In [21]:
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

print(cat_cols)

['complainant_type', 'department', 'campus_zone', 'building', 'floor', 'room_lab_no', 'facility_type', 'class_section', 'issue_category', 'issue_subcategory', 'severity', 'connectivity_impact', 'safety_risk', 'academic_impact', 'operational_impact', 'weather_condition', 'submission_channel']


C:\Users\acer\AppData\Local\Temp\ipykernel_21220\911867271.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns.tolist()


In [22]:
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])

In [23]:
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])

In [24]:
preprocessor = ColumnTransformer([("num", num_pipe, num_cols),("cat", cat_pipe, cat_cols)])

In [25]:
X_train_processed = preprocessor.fit_transform(X_train)

In [26]:
X_test_processed = preprocessor.transform(X_test)

In [27]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(160000, 308)
(40000, 308)


In [28]:
print(type(X_train_processed))
print(X_train_processed.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(160000, 308)


In [29]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(feature_names[:20])

308
['num__year_of_study' 'num__affected_users' 'num__issue_duration_hours'
 'num__previous_similar_complaints' 'num__complaints_last_7_days'
 'num__complaints_last_30_days' 'num__recurrence_count'
 'num__equipment_age_years' 'num__infrastructure_age_years'
 'num__estimated_repair_cost' 'num__complaint_month'
 'num__complaint_dayofweek' 'num__complaint_year' 'num__complaint_hour'
 'cat__complainant_type_Administrative Staff'
 'cat__complainant_type_Faculty' 'cat__complainant_type_Lab Staff'
 'cat__complainant_type_Maintenance Staff' 'cat__complainant_type_Student'
 'cat__department_Applied Sciences & Humanities']


## Preprocessing Summary

- Removed identifier column.
- Removed post-resolution leakage columns.
- Extracted useful date/time features.
- Separated target variable `priority`.
- Used stratified 80/20 train-test split.
- Numerical missing values are handled using median imputation.
- Categorical missing values are handled using most-frequent imputation.
- Categorical features are encoded using OneHotEncoder.
- Numerical features are standardized using StandardScaler.
- Preprocessing is implemented using ColumnTransformer and Pipeline.